# TCS Stock Price Prediction — All 11 ML Models
### Research Journey: OHLCV Only → Feature Engineering → OHLCV + Indicators
**Part A (Set A):** All 11 models on raw OHLCV (5 features)  
**💡 New Idea:** Add 9 Technical Indicators → Set B (14 features)  
**Part B (Set B):** All 11 models with indicators  
**Final:** Comprehensive comparison across all models & feature sets


## Step 0: Data Collection — Download TCS Stock Data from Yahoo Finance

**Source:** Yahoo Finance via `yfinance` library
**Stock:** TCS.NS (TCS listed on NSE — National Stock Exchange, India)
**Period:** 2005-01-01 to 2026-09-24
**Saved as:** TCS.csv

This is the raw data collection step. The downloaded CSV contains:
Open, High, Low, Close, Volume — daily historical data adjusted for stock splits and dividends.

In [1]:
pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
yf.download('TCS.NS', start='2005-01-01', end='2026-09-24').to_csv('TCS.csv')
print("TCS.csv downloaded and saved successfully!")

[*********************100%***********************]  1 of 1 completed

TCS.csv downloaded and saved successfully!


---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)


from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

print("All libraries imported successfully!")

---
## Step 2: Load Dataset

In [ ]:
df = pd.read_csv('TCS.csv', skiprows=[1, 2])
df.rename(columns={'Price': 'Date'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

for col in ['Open','High','Low','Close','Volume']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(inplace=True)
print("Dataset loaded!")
print(f"Rows: {len(df)}")

Dataset loaded!
Rows: 5363


## Step 3: Dataset Overview

In [ ]:
print(f"Shape: {df.shape}")
print(f"Date Range: {df.index[0].date()} to {df.index[-1].date()}")

Shape: (5363, 5)
Date Range: 2005-01-03 to 2026-09-23


In [ ]:
print("Data Types:")
print(df.dtypes)

Data Types:
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object


In [ ]:
df.head()

,Close,High,Low,Open,Volume
Date,,,,,
2005-01-03,108.272682,108.547908,106.848522,106.848522,4734168
2005-01-04,107.312592,108.331903,106.664522,108.011868,4979104
2005-01-05,106.011650,107.227780,102.435261,107.227780,9604320
2005-01-06,105.648438,107.691866,103.411394,105.611637,8006888
2005-01-07,107.208572,107.611820,105.024338,106.411688,7623136


In [ ]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [ ]:
print(f"Duplicate Rows: {df.duplicated().sum()}")

Duplicate Rows: 0


In [ ]:
print("Statistical Summary:")
df[['Open','High','Low','Close','Volume']].describe().round(2)

Statistical Summary:


,Open,High,Low,Close,Volume
count,5363.00,5363.00,5363.00,5363.00,5363.00
mean,1344.44,1357.71,1330.29,1343.67,3350520.35
std,1188.95,1199.08,1178.30,1188.29,2819411.07
min,75.80,78.58,69.43,75.49,0.00
25%,274.27,277.70,270.58,274.01,1853218.00
50%,959.55,967.05,951.05,958.63,2650784.00
75%,2383.67,2414.17,2349.66,2376.28,3987019.00
max,4251.38,4266.48,4191.92,4230.71,88067154.00


---
## Step 4: EDA Graphs